In [ ]:
"B2, B3, B4, B8, B11, B12, SCL, NDVI, BSI"

In [ ]:
import ee

# =====================================================
# 0. BOUNDING BOX LIST (Stonehenge + Norfolk)
# =====================================================
boxes = [
            ee.Geometry.Rectangle([-1.875, 51.145, -1.775, 51.205]),
            ee.Geometry.Rectangle([-1.895, 51.405, -1.815, 51.455]),
            ee.Geometry.Rectangle([-1.895, 51.31,  -1.85,  51.338]),
            ee.Geometry.Rectangle([-1.12,  51.335, -1.05,  51.38]),
            ee.Geometry.Rectangle([1.255,  52.565, 1.325,  52.6]),
            ee.Geometry.Rectangle([1.275,  52.608, 1.322,  52.635])
]

Map.centerObject(boxes[0], 11);
boxes.forEach(function(b, i) {
  Map.addLayer(b, {color: 'red'}, 'Box ' + i);
});

# =====================================================
# 1. SENTINEL‑2 (OPTICAL)
# =====================================================
s2_bands = ["B2","B3","B4","B8","B11","B12","SCL","NDVI","BSI"];

  - B2
  - B3
  - B4
  - B8
  - B11
  - B12
  - SCL
  - NDVI
  - BSI

def maskS2_SCL(image):
#   scl = image.select('SCL');
#   mask = scl.neq(3).and(scl.neq(8))
#                 .and(scl.neq(9)).and(scl.neq(10))
#                 .and(scl.neq(11));
  
  return image.updateMask(mask);


def loadS2(aoi):
  s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    # .filterBounds(aoi)
    # .filterDate("2018-01-01", "2024-12-31")
    # .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    # .select(s2_bands);

  return s2.map(maskS2_SCL).median().clip(aoi)

# =====================================================
# 2. SENTINEL‑1 (SAR)
# =====================================================
def loadS1(aoi) :
  return ee.ImageCollection("COPERNICUS/S1_GRD")
    # .filterBounds(aoi)
    # .filterDate("2018-01-01", "2024-12-31")
    # .filter(ee.Filter.eq("instrumentMode", "IW"))
    # .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
    # .filter(ee.Filter.eq("resolution_meters", 10))
    # .select(["VV", "VH"])
    # .median()
    # .clip(aoi)


# =====================================================
# 3. BUILD STACK FOR A BOX
# ====================================================
def buildStack(aoi):
  s2 = loadS2(aoi)
  s1 = loadS1(aoi)

  ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI");

  bsi = s2.expression(
    "(SWIR + RED - NIR - BLUE) / (SWIR + RED + NIR + BLUE)", {
      "SWIR": s2.select("B11"),
      "RED":  s2.select("B4"),
      "NIR":  s2.select("B8"),
      "BLUE": s2.select("B2")
    }).rename("BSI")

  return s2.addBands(s1).addBands(ndvi).addBands(bsi).toFloat();


# =====================================================
# 4. EXPORT LOOP
# =====================================================
#boxes.forEach(function(aoi, i) {

  print("Processing tile:", i)

  stack = buildStack(aoi)

  # =====================================================
  # FAST VALIDITY CHECK (no memory issues)
  # =====================================================
  # =====================================================
  # FAST VALIDITY CHECK (no memory issues)
  # =====================================================
#   anyValue = stack.select('B2')
#     .reduceRegion({
#       reducer: ee.Reducer.anyNonZero(),
#       geometry: aoi,
#       scale: 30,
#       maxPixels: 1e9
#     })
#     .get('B2');  // server-side object

  // Convert to a server-side boolean: true if non-null and non-zero
  var hasData = ee.Algorithms.IsEqual(anyValue, 1);

  // Bring to client to decide whether to export
  if (!hasData.getInfo()) {
    print('Skipping empty tile', i);
    return;
  }

  // =====================================================
  // METADATA
  // =====================================================
  var meta = ee.Feature(null, {
    tile_id: i,
    bbox: aoi.bounds(1).coordinates(),
    projection: stack.projection().crs(),
    scale_m: stack.projection().nominalScale(),
    bands: stack.bandNames()
  });

  // =====================================================
  // EXPORT METADATA
  // =====================================================
  Export.table.toDrive({
    collection: ee.FeatureCollection([meta]),
    description: 'tile_' + i + '_metadata',
    folder: 'stonehenge_dataset',
    fileNamePrefix: 'tile_' + i + '_metadata',
    fileFormat: 'JSON'
  });

  // =====================================================
  // EXPORT IMAGE
  // =====================================================
  Export.image.toDrive({
    image: stack,
    description: 'tile_' + i,
    folder: 'stonehenge_dataset',
    fileNamePrefix: 'tile_' + i,
    region: aoi,
    scale: 10,
    maxPixels: 1e13
  });

});
